# 软件工程智能体：循环、评测与接口

> 前面几讲实现的 Agent 循环、验证器、工具执行，大多停留在合成任务上：数学应用题、检索问答、迷宫导航。这一讲把同一套循环搬到真实软件工程任务上，让 Agent 修一个真实仓库里的 bug，或者写一段高性能内核。真实软件工程和解题不一样：代码库动辄上百万 token，反馈也来自真实系统，编译错误、测试失败、性能画像都是现成的信号。
>
> 这一讲按三个递进的问题展开。第一，第 2 讲反复用过一种做法：训练结束后不动权重，回答问题时多花算力，它叫 test-time compute。在代码任务上，这个做法还能买到多少准确率。第二，在"对"之外还要"快"时，评测该怎么设计。第三，Agent 与底层系统之间靠什么接口对齐。最后把三块拼成一个能修真实 bug 的迷你循环。

你可能见过这样的现象。GitHub 上有不少 PR 标注着由 AI 生成，Cursor、Copilot 这类编程助手也能在你写代码时自动补全，甚至直接帮你改一段出错的代码。它们在做的是同一个循环：读代码、改代码、跑测试、拿反馈、再改。学完这一讲，你能亲手搭出这样一个循环，让 Agent 修真实仓库里的 bug、写高性能内核。这也是第 1 讲感知、决策、行动、反馈的框架，第一次用在"改代码"这个真实任务上。

改真实仓库比前面解题难。仓库往往有几百个文件，先要找出 bug 藏在哪。改完要跑测试确认，测试通过不等于改对，还可能改坏别的功能。还要生成多个候选修改，再从中挑出正确的。软件工程因此被普遍认为是 Agent 最有价值的应用场景之一。

先看怎么度量"AI 修 bug 修得好不好"。SWE-bench 收集真实 GitHub 仓库里的真实 issue，配上能自动判定对错的测试，Agent 输出一个补丁，跑测试看是否通过；SWE-bench Verified 是其中经过人工核验的一批题，数字更可信。CodeMonkeys 是本讲的主角，它把"修一个 issue"切成三段——找上下文、生成候选、选择答案——每段配一个独立度量。一个数字说明"选择"有多难：从候选里随机挑一个提交，正确率只有 45.8%，而一个总能选对的 oracle 能到 69.8%，正确答案就在候选池里，系统却选不出来。第一节从生成段开始，看 test-time compute 在代码任务里能买到多少覆盖率。

## 1. 代码任务中的推理期算力

这一节回答两个问题：多生成几份候选补丁，修对的比例能升多少；给定一笔预算，在串行和并行之间怎么分。

第 2 讲我们在数学题上见过同样的思路：重复采样能把修对的比例推向 1，且随采样数近似对数线性增长。代码任务保留了这条规律，但多一层复杂性——修好一个 issue 不是输出一个答案，而是要产出能在官方测试上通过的代码修改。CodeMonkeys 把"修一个 issue"拆成三个可分别缩放的子任务，本节先处理生成段：修对的比例怎么随采样数增长。

这一节要引入一个度量，叫覆盖率。覆盖率的意思是一批题目里，至少有一条候选补丁修对的题占多大比例。它衡量的是"生成段"的上限——候选池里到底有没有正确答案。

先手算一个两题的小例子。设单条候选补丁修对的概率为 p，独立采样 n 条。每条都修错的概率是 $(1-p)^n$，所以至少有一条修对的概率是 $1-(1-p)^n$。n 取 2 和 5：

| 题 | 单条补丁正确率 p | n=2 的通过率 | n=5 的通过率 |
|---|---|---|---|
| 题 1 | 0.2 | $1-0.8^2=0.36$ | $1-0.8^5\approx 0.67$ |
| 题 2 | 0.9 | $1-0.1^2=0.99$ | $1-0.1^5\approx 1.00$ |

覆盖率是两题通过率的平均：n=2 时 $(0.36+0.99)/2=0.675$，n=5 时约 $0.835$。同一批题，只增加采样数，覆盖率就从 0.675 涨到 0.835。下面用代码验证这两个数，再生成一批难易不同的题目，观察覆盖率随采样数的增长。

In [ ]:
import numpy as np

# 两道题的通过率：p1=0.2, p2=0.9，独立采样 n 条，覆盖率取两道题的平均
p = np.array([0.2, 0.9])
for n in (2, 5):
    per_issue = 1 - (1 - p) ** n
    print("n=%d  单题通过率 %s  覆盖率 %.3f" % (n, per_issue, per_issue.mean()))
print("手算结果一致：n=2 覆盖 0.675，n=5 覆盖约 0.836。")

In [ ]:
# 一批难易不同的题：单条补丁正确率 p_i 服从右偏的 Beta 分布
rng = np.random.default_rng(42)
n_problems = 200
p_i = rng.beta(1.5, 5.0, size=n_problems)
print("p_i 均值 %.3f、中位数 %.3f：多数题单条补丁就做不对" % (p_i.mean(), np.median(p_i)))

samples = np.arange(1, 51)
coverage = np.array([(1 - (1 - p_i) ** k).mean() for k in samples])
print("k=1 覆盖率 %.3f，k=50 覆盖率 %.3f" % (coverage[0], coverage[-1]))

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(samples, coverage)
ax.set_xscale("log")
ax.set_xlabel("sampled patches k")
ax.set_ylabel("coverage")
ax.set_title("Coverage grows with parallel sampling")
plt.show()
print("关键观察：覆盖率随采样数上升，前几次采样收益最大，之后增速放缓。")

上面的实验只用了并行采样：每条候选补丁独立生成，互不影响。真实系统还会用串行修复：把算力反复花在同一条轨迹上，生成补丁、跑测试、拿反馈、再生成，多轮迭代。轨迹是这里的一个术语，指一条从头修到尾的尝试。真实系统两种都用，所以问题变成：给定一笔预算，在串行和并行之间怎么分。

我们用一个简化模型看这个分配。总预算 B 次调用，拆成 N 条轨迹，每条轨迹做 r 轮修复。单条轨迹初次生成就对的概率是 p，做错之后每轮修复以概率 $\eta$ 补对。下面先手算这个模型，再用代码观察同一预算下不同拆分的覆盖率。

先看一次调用对应什么。一次调用就是一次生成。修复是生成之后的追加动作：给定总预算 B 次调用，拆成 N 条轨迹、每条轨迹做 r 轮修复，那么每条轨迹实际产生 r+1 次调用——第 0 次是初次生成，之后 r 轮修复各一次。这就是约束 $N(r+1)=B$ 的来历。

单条轨迹的解出概率分两步算。初次生成以概率 p 直接做对；做错（概率 $1-p$）才进入修复，每轮修复独立地以概率 $\eta$ 补对，继续错的概率是 $1-\eta$。r 轮全错的概率是 $(1-\eta)^r$，所以

$$\text{serial\_success} = 1-(1-p)(1-\eta)^r.$$

手算一组数字。取 p=0.2，$\eta=0.3$：

| 修复轮数 r | 单条轨迹解出概率 |
|---|---|
| 0 | $1-0.8=0.20$ |
| 1 | $1-0.8\times0.7=0.44$ |
| 4 | $1-0.8\times0.7^4\approx0.81$ |

r=0 就是纯并行采样，没有修复，解出概率等于初次生成概率 p。r 从 0 变到 4，同一笔调用摊在一条轨迹上，解出概率从 0.20 升到约 0.81，修复让同一笔预算产出更高。代价是轨迹数变少：B=10 时，r=4 只能开 $10/5=2$ 条并行轨迹。预算分配的本质，是在"每条轨迹更聪明"和"更多条轨迹"之间权衡。

In [ ]:
def serial_success(p, rounds, eta):
    """单条轨迹做 rounds 轮串行修复后的解出概率。

    p：单次生成正确的概率；eta：每轮修复补对的概率。
    初次失败 (1-p)，之后每轮以 (1-eta) 继续失败。
    """
    return 1 - (1 - p) * (1 - eta) ** rounds

def split_coverage(p_i, n_traj, rounds, eta):
    """n_traj 条并行轨迹、每条 rounds 轮修复后的覆盖率。"""
    s = serial_success(p_i, rounds, eta)
    return 1 - (1 - s) ** n_traj

# 三条固定拆分：纯并行 / 浅修复 / 深修复，比较随总预算的增长
splits = {"parallel (r=0)": 1, "shallow (r=1)": 2, "deep (r=4)": 5}
budgets = np.arange(1, 41)
fig, ax = plt.subplots(figsize=(7, 4))
for label, calls in splits.items():
    covs = [split_coverage(p_i, max(1, B // calls), calls - 1, 0.3).mean()
            for B in budgets]
    ax.plot(budgets, covs, label=label)
ax.axhline(p_i.mean(), color="gray", ls="--", label="single generation")
ax.set_xlabel("total calls B")
ax.set_ylabel("coverage")
ax.set_title("Serial vs parallel budget allocation")
ax.legend(fontsize=8)
plt.show()

for label, calls in splits.items():
    cov = split_coverage(p_i, max(1, 10 // calls), calls - 1, 0.3).mean()
    print("%-16s B=10 覆盖率 %.3f" % (label, cov))
print("关键观察：预算紧张时带修复的拆分更划算，预算充足后各拆分趋同；"
      "单次生成停在低水平。")

覆盖率是生成段的天花板。即使选择阶段有一个 oracle——一个总能选对的假想选择器——最终得分也超不过覆盖率，因为候选池里没有正确答案时，谁都没得选。CodeMonkeys 的三个数字把这条链讲得很清楚：候选池的覆盖率是 69.8%，随机选择只有 45.8%，oracle 选择恰好等于覆盖率。选择方法决定从天花板上收回多少。下面在合成候选池上实现几种选择器，量化各自的回收比例。

合成数据的口径：300 道题，每题 8 条候选补丁。每条候选有一个隐藏的正确性标签，还被 5 个"生成测试"检查过——正确候选单次测试通过率较高，错误候选偶尔也会蒙混过关。这和真实系统一致：生成测试并不完美，正是这种不完美让选择变得困难。

把覆盖与选择两阶段放在一个具体问题上走一遍，比只看公式清楚。题目：把 compute_total(n)（返回 0 到 n 的和）改对且改快。生成段产出了 4 条候选补丁：

| 候选 | 实现思路 | 正确性 | 速度 |
|---|---|---|---|
| A | 等差求和 $n(n+1)/2$ | 对 | O(1)，快 |
| B | 求和公式漏项 $n(n-1)/2$ | 错 | O(1)，快 |
| C | 恒返回 0 | 快但错 | O(1)，最快 |
| D | 逐个累加 sum(range(n+1)) | 对 | O(n)，慢 |

覆盖阶段只问一件事：池子里有没有正确答案。A 和 D 都对，所以这一池的覆盖成立。换几批池子看覆盖怎么筛。池 1 有 {A, D}，池 2 只有 {B, C}，池 3 有 {A, B}。池 2 没有正确候选，这一池无论选择阶段多强都解不出来，只能从覆盖统计里被剔除。覆盖率就是"有多少池子至少含一条正确候选"。CodeMonkeys 里 oracle 选择恰好等于覆盖率，原因就在这里：oracle 只在池子有正确答案时才有得选。

选择阶段根据弱信号从池子里挑一条。假设生成测试只有两条，都只断言正确性，看不见快慢：

| 候选 | 测试 1：compute_total(0)==0 | 测试 2：compute_total(100)==5050 | 通过数 |
|---|---|---|---|
| A | 过（0 恰为 0） | 过（5050 恰好） | 2 |
| B | 过（公式给出 0） | 不过（得 4950） | 1 |
| C | 过（恒 0） | 不过 | 1 |
| D | 过 | 过 | 2 |

投票结果 A、D 各 2 票，测试信号区分不出"对且快"与"对但慢"。这正是后面两节要解决的：fast_p 用加速比给正确候选继续分层，接口设计把速度信号显式暴露给 Agent。在二元测试下，选择器只能靠补丁长度、复杂度这类启发式在 A、D 之间定夺，下面代码里的 argmax 取先出现的 A。

随机选择从 4 条里均匀挑，命中率 1/2；测试投票把答案缩到两条正确的候选；若池子里只有 {B, C}，覆盖率已经归零。两阶段分开度量，选择损失才能被量化：CodeMonkeys 的 45.8% 与 69.8% 之差，就是选择方法从覆盖天花板上丢掉的。

In [ ]:
# 合成候选池：300 题 × 8 条候选 × 5 个生成测试
M, C, T = 300, 8, 5
rng = np.random.default_rng(7)
q = rng.beta(1.2, 6.0, size=M)                # 每题含正确候选的比例
correct = rng.random((M, C)) < q[:, None]      # 隐藏的正确性标签
test_result = np.where(correct[:, :, None],
                       rng.random((M, C, T)) < 0.75,   # 正确候选测试通过率高
                       rng.random((M, C, T)) < 0.20)   # 错误候选偶尔蒙混过关

def random_score(correct):
    """随机挑一条候选，猜中正确候选的比例。"""
    pick = rng.integers(0, correct.shape[1], size=correct.shape[0])
    return correct[np.arange(correct.shape[0]), pick].mean()

def vote_score(correct, test_result):
    """测试多数投票：挑通过生成测试最多的候选。"""
    votes = test_result.sum(axis=2)
    pick = np.argmax(votes, axis=1)
    return correct[np.arange(correct.shape[0]), pick].mean()

def oracle_score(correct):
    """oracle 选择：存在正确候选就必选对，即覆盖率。"""
    return correct.any(axis=1).mean()

cov = oracle_score(correct)
rand = random_score(correct)
vote = vote_score(correct, test_result)
print("随机选择   %.3f" % rand)
print("测试投票   %.3f" % vote)
print("oracle 选择 %.3f（= 覆盖率）" % cov)
print("投票收回随机与天花板之间的 %.0f%%" % ((vote - rand) / (cov - rand) * 100))
print("关键观察：合成数据里测试近乎无噪声，投票接近天花板；"
      "真实系统里测试更弱、候选结果高度相关，回收比例低得多。")

## 2. 用测试和性能反馈改进代码

上一节的评测是二元的：补丁修好了 issue 就算对。但很多工程任务的成败不是二元——函数既要正确，又要快。这一节解决这个问题：怎么同时度量"对"和"快"，以及怎么让 Agent 把函数改得又快又对。

我们用一个基准叫 KernelBench。它的任务给模型一个 PyTorch 参考实现，要求模型输出一个接口相同、内部换成自定义 CUDA 内核的版本，然后自动评测两条轴：功能正确（随机输入对比输出）和性能（相对 PyTorch 的加速比）。CUDA 是 GPU 上的编程语言，内核是 GPU 上执行的一段程序，这里的参考实现就是用 PyTorch 这个常用深度学习框架写的。250 个任务按算子数分成三级，Level 2 考的是把一串算子融合成一个内核。本节先把"正确且快"压缩成一个标量指标 fast_p，再看测试时方法怎么提升它。

fast_p 算的是：在一批候选内核里，同时满足"正确"和"加速比大于阈值 p"的候选占多大比例。设 $p$ 为加速比阈值，$N$ 个候选内核中，这个占比就是

$$\text{fast}_p = \frac{1}{N}\sum_{i=1}^{N} \mathbb{1}[\text{correct}_i \land \text{speedup}_i > p].$$

取 $p=0$ 时它就是纯正确率；$p$ 每升高一档，就多筛掉一批"对但不够快"的候选。手算四个候选：

| 候选 | 正确 | 加速比 | 进 fast_0 | 进 fast_1 | 进 fast_2 |
|---|---|---|---|---|---|
| A | 是 | 3.2 | 是 | 是 | 是 |
| B | 是 | 0.8 | 是 | 否 | 否 |
| C | 否 | 5.0 | 否 | 否 | 否 |
| D | 是 | 1.5 | 是 | 是 | 否 |

fast_0 = 3/4，fast_1 = 2/4，fast_2 = 1/4。B 正确但慢，进得了 fast_0 出不了 fast_1；C 快但错，连 fast_0 都进不了。两个候选各卡在一条轴上，这正是"正确且快"两个条件缺一不可的原因。

fast_p 的公式里有两个记号需要读顺。$\mathbb{1}[\cdots]$ 是指示函数，方括号里的条件成立时取 1，否则取 0。$\land$ 是逻辑与，两个条件必须同时成立。整句读出来是：在所有 N 条候选里，数出"既正确、又比 PyTorch 快过 p 倍"的条数，再除以 N。

把它想成两重闸门，第一道查正确性，第二道查加速比。候选 A（正确、3.2 倍）依次通过两道。B（正确、0.8 倍）过了第一道，第二道要求快过 1 倍，0.8 不够，掉在这里。C（错、5.0 倍）速度很高，但第一道就没过。D（正确、1.5 倍）两道都过，可 p 抬到 2 时，1.5 又不够了。阈值 p 决定第二道闸门的高度——p=0 时这道闸门几乎放行，fast_0 退化成纯正确率。

条件写的是 $>p$ 而不是 $\ge p$。加速比恰好等于 p 的候选不计入。p=1 意味着"严格快过基线"，恰好和 PyTorch 一样快不算达标。这是一个边界口径，影响的只是恰好等于阈值的候选，通常可以忽略。

把"正确"与"快"压成一个标量，是为了让评测得到单一数字来排序。二元评测（对或错）无法区分"对但慢"与"对且快"，KernelBench 必须用加速比给"对"的候选继续分层。p 是评测的旋钮：调高变难，调低变松，同一批候选在不同 p 下得到一条 fast_p 曲线，方便在不同严格度下对比模型与方法。

In [ ]:
def fast_p(candidates, p):
    """返回正确且加速比大于 p 的候选占比。

    candidates：形如 (is_correct, speedup) 的列表。
    """
    correct = np.array([c[0] for c in candidates], dtype=bool)
    speedup = np.array([c[1] for c in candidates])
    return np.mean(correct & (speedup > p))

cands = [("A", True, 3.2), ("B", True, 0.8), ("C", False, 5.0), ("D", True, 1.5)]
pairs = [(c, s) for _, c, s in cands]
for p in (0.0, 1.0, 2.0):
    print("fast_%.1f = %.2f" % (p, fast_p(pairs, p)))
print("关键观察：p 提高一档，就多筛掉一批'对但慢'的候选。")

In [ ]:
# 合成一批候选内核：正确与否随机，加速比按对数正态分布
rng = np.random.default_rng(3)
n = 300
correct = rng.random(n) < 0.4
speedup = np.exp(rng.normal(0.2, 0.9, size=n))
pairs = list(zip(correct, speedup))

ps = np.linspace(0.0, 2.5, 101)
fasts = np.array([fast_p(pairs, p) for p in ps])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ps, fasts)
ax.axvline(1.0, color="gray", ls="--", label="p = 1")
ax.set_xlabel("speedup threshold p")
ax.set_ylabel("fast_p")
ax.set_title("fast_p shrinks as the threshold rises")
ax.legend()
plt.show()
print("fast_0=%.3f  fast_1=%.3f  fast_2=%.3f"
      % (fast_p(pairs, 0.0), fast_p(pairs, 1.0), fast_p(pairs, 2.0)))
print("关键观察：阈值 p 是评测的旋钮，调高就变难，"
      "这条曲线也便于将来把基线升级到 torch.compile。")

先把目标定清楚：让模型写一个又快又对的内核，一次就成的概率有多低。one-shot 指只让模型生成一次。在 KernelBench 上，前沿模型 one-shot 平均只有不到 20% 的任务能快过 PyTorch Eager——PyTorch 默认的逐算子执行方式。写对已经很难，写对还快更难。

多花推理算力的方法（也就是 test-time compute）把预算花在多次调用上，这是第 2 讲的老思路。KernelBench 对比了两种。一种是重复采样：并行生成 k 条候选，只要一条达标就算成功。另一种是迭代优化：多轮把上一轮的生成结果 G、执行反馈 E、profiler 反馈 P 喂回模型，每轮都比上一轮多知道一点。同为 10 次调用预算，迭代更优：DeepSeek-R1 在 Level 2 的 fast_1（阈值 p=1，即严格比 PyTorch 快）从 one-shot 的 36% 升到 72%，反馈组合 G+E+P 最强。下面用一个简化模型看反馈如何抬高每轮的成功率。

三个字母各指什么。G（generation）是上一轮生成的内核代码。E（execution）是上一轮运行的结果，包括对不对、花了多少毫秒。P（profiling）来自 profiler——测量程序时间花在哪的工具，逐行报告每条语句占用的时间比例。Python 自带的 cProfile 就是这类工具，GPU 上有 NVIDIA 的 ncu 一类工具。

用一个具体例子看三份反馈怎么接力。任务是写一个函数返回数组的最大值。

- 轮 1 生成实现 G₁：两层循环，对每个元素再扫描一遍全数组找最大，复杂度 O(n²)。执行反馈 E₁：结果正确，但 n=10000 时耗时数秒。profiler 反馈 P₁：第 5 行的内层扫描占 99% 时间。
- 轮 2 模型读到 P₁，意识到内层扫描是冗余的，生成 G₂：一遍线性扫描 O(n)。E₂：结果正确，耗时降到毫秒级。

每一轮喂回去的信息都让下一轮离"正确且快"更近一步，单轮成功率从 p₀ 向 p_max 爬升。重复采样没有这个机制：k 次独立采样每次成功率都是同一个 p，水平不会随调用次数提高。这就是同预算下迭代更强的来源——DeepSeek-R1 在 Level 2 的 fast_1 从 one-shot 的 36% 升到 72%，靠的是每轮比上一轮多知道一点。

下面用简化模型量化爬升：每轮成功率按一条饱和曲线从 p₀ 逼近 p_max，累计覆盖率取"任意一轮成功"的概率。反馈组合越丰富（E+P），p_max 越高。

In [ ]:
def repeated_coverage(p, k):
    """k 次独立采样的覆盖率。"""
    return 1 - (1 - p) ** k

def iterative_cum(p0, p_max, rounds, tau=3):
    """迭代优化的累计覆盖率：第 t 轮成功率从 p0 向 p_max 爬升。

    每轮成功概率 p_t = p0 + (p_max - p0) * (1 - exp(-t/tau))，
    累计覆盖率为任意一轮成功的概率。
    """
    t = np.arange(1, rounds + 1)
    p_t = p0 + (p_max - p0) * (1 - np.exp(-t / tau))
    return 1 - np.cumprod(1 - p_t)

B = 10
ks = np.arange(1, B + 1)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ks, repeated_coverage(0.20, ks), label="repeated sampling")
ax.plot(ks, iterative_cum(0.20, 0.45, B), label="iterative (E)")
ax.plot(ks, iterative_cum(0.20, 0.60, B), label="iterative (E+P)")
ax.axhline(repeated_coverage(0.03, B), color="gray", ls=":", label="hard group")
ax.set_xlabel("calls")
ax.set_ylabel("fast_1 coverage")
ax.set_title("Execution and profiler feedback lift success")
ax.legend(fontsize=8)
plt.show()

print("容易题 10 次调用：重复采样 %.3f，加执行反馈 %.3f，加 profiler %.3f"
      % (repeated_coverage(0.20, B), iterative_cum(0.20, 0.45, B)[-1],
         iterative_cum(0.20, 0.60, B)[-1]))
print("难题 10 次调用：重复采样 %.3f，加执行反馈 %.3f"
      % (repeated_coverage(0.03, B), iterative_cum(0.03, 0.05, B)[-1]))
print("关键观察：反馈抬高每轮成功率，迭代因此强于重复采样；"
      "但内在成功率过低的题目，再多反馈也无济于事。")

## 3. 让 Agent 的动作符合系统接口

前两节把注意力放在 Agent 的生成与选择上，隐含了一个假设：Agent 读到的反馈就是系统的真实语义。这个假设经常不成立。这一节要解决的问题是：当接口展示的信息和系统真实语义不一致时，Agent 会怎么犯错，又该怎么对齐。

先给这个现象起个名字。论文 On the Need to Align Intent and Implementation 把它叫做 construct drift：为 A 算出来的不确定度，被拿来支撑关于 B 的结论。工程上的形态，就是把测试通过率当作修复成功、把高覆盖率当作高正确率。Agent 的意图必须与接口的实现语义对齐，否则指标会漂移。

落到系统层，论文 Agent-System Interface 给出两个手段。DSL（领域专用语言）把"怎么映射"显式化成一个可搜索的空间。AutoGuide 把原始执行输出翻译成可行动的自然语言建议。下面用一个最小例子演示接口不匹配时 Agent 的失败模式。

接口在这里指 Agent 每次行动后能读到的信息集合。Agent 只能根据读到的内容做决策，读不到的信息对它来说等于不存在，接口决定了 Agent 眼中的世界。当接口展示的信息与系统真实语义不一致时，指标就会漂移，这就是前面提到的 construct drift。

先看接口不同，行为怎么不同。两个 Agent 各带一个接口修同一个 bug。接口 1 只返回"通过了 3 个测试中的 1 个"，Agent 只能知道还差 2 个，无从判断错在哪。接口 2 返回失败的断言本身——"sum_evens([1, 2, 3, 4]) 期望 6 得到 4"，Agent 直接看到错误值。同一个系统状态，接口 2 给的信息严格更多，下一步动作自然不同。错位还有一种更隐蔽的形态：接口说"通过"，语义却是"没有崩溃"而非"结果正确"。测试只检查函数不抛异常、不检查返回值时，任何不崩溃的实现都算通过，模型以为修好了，结果还是错的。

接口论文给出两个手段。第一个是 DSL，领域专用语言。与其让模型自由发挥，不如给它一套固定的映射原语，把可行改动压缩成一个可搜索的空间。KernelBench 允许使用 Triton 就属于这类。Triton 是一种面向 GPU 的高层编程语言，它的分块和共享内存由编译器隐式处理，模型不必自己管理这些细节，写出来的内核也更容易被编译通过。第二个是 AutoGuide，处理反向的信息流：把原始的 profiler 输出——几万行采样报告——翻译成可行动的自然语言建议，比如"第 3 行的循环占 80% 时间，把取余运算提到循环外"。模型读到的是人话，不是原始采样栈。

下面用一个最小例子演示接口错位：同一个 Agent，只换接口，行为就从"挑最快的"变成"挑又快又对的"。

设计一个最小的接口错位。任务：把 compute_total(n)（返回 0 到 n 的和）改快，且必须保持正确。仓库里有三个候选实现：朴素版（O(n)）、只快不正确的作弊版（丢弃输入、恒返回 0）、又快又正确的公式版（等差求和）。两套接口：接口 A 只报告耗时文本，接口 B 报告结构化结果——正确性、耗时、相对基线的加速比。Agent 的策略很简单：按接口报告挑"最好"的候选。接口决定了它把哪个目标当成"好"。

In [ ]:
import time

def compute_total_naive(n):
    return sum(range(n + 1))

def compute_total_fast_wrong(n):
    return 0                     # 丢弃输入：耗时接近 0，但结果错误

def compute_total_fast_right(n):
    return n * (n + 1) // 2      # 等差求和：正确且 O(1)

proposals = [("naive", compute_total_naive),
             ("fast_wrong", compute_total_fast_wrong),
             ("fast_right", compute_total_fast_right)]

def measure_ms(fn, n=20000, repeat=50, runs=5):
    """多次计时取中位数，返回一次调用的平均毫秒数。"""
    samples = []
    for _ in range(runs):
        t0 = time.perf_counter()
        for _ in range(repeat):
            fn(n)
        samples.append((time.perf_counter() - t0) / repeat * 1000)
    return float(np.median(samples))

def harness_a(fn, n):
    """接口 A：只返回耗时文本，不暴露正确性。"""
    return "%.3f ms" % measure_ms(fn, n)

def harness_b(fn, n, reference):
    """接口 B：结构化报告，给出正确性、耗时与相对基线加速比。"""
    ms = measure_ms(fn, n)
    baseline = measure_ms(reference, n)
    return {"correct": fn(n) == reference(n),
            "time_ms": ms,
            "speedup": baseline / ms if ms > 0 else float("inf")}

def agent_under_a(proposals, n):
    """接口 A 下按耗时文本挑候选：取最小值，平手取先出现者。"""
    best_name, best_text = None, None
    for name, fn in proposals:
        text = harness_a(fn, n)
        if best_text is None or text < best_text:
            best_name, best_text = name, text
    return best_name, best_text

def agent_under_b(proposals, n, reference):
    """接口 B 下先筛正确，再按加速比挑候选。"""
    best_name, best_speed = None, -1.0
    for name, fn in proposals:
        r = harness_b(fn, n, reference)
        if r["correct"] and r["speedup"] > best_speed:
            best_name, best_speed = name, r["speedup"]
    return best_name, best_speed

n = 20000
for name, fn in proposals:
    print("%-12s %.4f ms" % (name, measure_ms(fn, n)))

name_a, text_a = agent_under_a(proposals, n)
name_b, speed_b = agent_under_b(proposals, n, compute_total_naive)
correct_a = dict(proposals)[name_a](n) == n * (n + 1) // 2
correct_b = dict(proposals)[name_b](n) == n * (n + 1) // 2
print("接口 A 选中 %-11s 外部判定正确 = %s" % (name_a, correct_a))
print("接口 B 选中 %-11s 外部判定正确 = %s（加速比 %.1fx）" % (name_b, correct_b, speed_b))
print("关键观察：接口 A 把'快'和'对'压缩进一个耗时文本，"
      "Agent 没有任何正确性信号；接口 B 显式暴露正确性，"
      "Agent 的意图才与系统语义对齐。")

## 4. SWE-Agent 循环：两个状态机的修复

前三节是拼图的三块：test-time compute 回答预算怎么花，评测指标回答目标怎么定，接口回答反馈怎么对齐。这一节把三块组装成一个能在玩具仓库上运转的完整循环。目标是把"找上下文、生成候选、选择答案"里最核心的生成候选做成真实代码：读取源码、修改、跑测试、拿反馈、重试。

为了把反馈利用到极致，CodeMonkeys 把修复拆成两个背靠背的状态机。状态机在这里指两个分工不同的循环。第一个循环先生成一个能复现 issue 的测试，第二个循环拿这个测试当判官，让编辑与测试互相纠错。下面先搭玩具仓库，再做测试状态机，最后做编辑状态机。

先用一个可控的玩具仓库，比直接上真实仓库更容易看清结构。仓库里有一个模块 buggy.py，里面有一个带缺陷的函数 sum_evens(nums)，它想把列表里的偶数加起来，却把奇数也累加进了总和。判定对错可以程序化：正确实现是固定的，把输入代入就行。

测试状态机的目标是产出这样一个测试脚本：它在未修复的 buggy.py 上必须失败，在参考实现上必须通过。前一条叫复现 issue——测试失败说明 bug 确实被触发；后一条叫两侧检查，确保测试真的判得出对错。编辑状态机拿这个测试当判官，迭代修改源码。整个循环需要 LLM 提供两种产物：测试脚本和修改后的源码。

In [ ]:
import sys, os
_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, 'llm_client.py')):
    _root = os.path.dirname(_root)
    if _root == os.path.dirname(_root):
        break
if _root not in sys.path:
    sys.path.insert(0, _root)
from llm_client import get_llm
client = get_llm()

if False:
    print("LLM 客户端就绪，真实 API 演示：测试与编辑由脚本化轨迹提供，"
          "循环骨架与真实 API 一致。")
else:
    print("LLM 客户端就绪，真实 API 模式。")

In [ ]:
import tempfile

repo_dir = tempfile.mkdtemp(prefix="swe_repo_")
BUGGY = ('def sum_evens(nums):\n'
         '    """返回列表中所有偶数的和。"""\n'
         '    total = 0\n'
         '    for x in nums:\n'
         '        if x % 2 == 1:      # 缺陷：把奇数也累加进总和\n'
         '            total += x\n'
         '    return total\n')
FIXED = ('def sum_evens(nums):\n'
         '    """返回列表中所有偶数的和。"""\n'
         '    total = 0\n'
         '    for x in nums:\n'
         '        if x % 2 == 0:      # 修正：只累加偶数\n'
         '            total += x\n'
         '    return total\n')

with open(os.path.join(repo_dir, "buggy.py"), "w") as f:
    f.write(BUGGY)

print("toy 仓库目录:", repo_dir)
print(BUGGY)

In [ ]:
import subprocess

def run_test(repo_dir, test_source, max_out=3):
    """在仓库目录里执行一段测试脚本，返回 (exit_code, 输出尾部)。

    repo_dir：仓库目录；test_source：测试脚本源码，脚本通过
    import 仓库里的模块来验证行为。断言失败时 exit code 非 0。
    """
    test_path = os.path.join(repo_dir, "_run_test.py")
    with open(test_path, "w") as f:
        f.write(test_source)
    proc = subprocess.run(
        [sys.executable, "_run_test.py"], cwd=repo_dir,
        capture_output=True, text=True, timeout=30,
    )
    tail = (proc.stdout + proc.stderr).strip().splitlines()[-max_out:]
    return proc.returncode, tail

# 在未修复的 buggy.py 上跑一段代表测试，应复现 issue（exit 非 0）
code, tail = run_test(repo_dir, "from buggy import sum_evens\n"
                                 "assert sum_evens([1, 2, 3, 4]) == 6\nprint('PASS')")
print("未修复代码上 exit=%d" % code)
print("输出尾部:", tail)
print("关键观察：退出码非 0 就是最廉价的执行反馈。")

把串行修复循环在 sum_evens 上逐步走一遍。先读代码：buggy.py 里的 sum_evens(nums) 想累加偶数，条件却写成了 `x % 2 == 1`。`%` 是取余，`x % 2 == 1` 在 x 是奇数时成立，于是函数把奇数也加进了 total。对 [1, 2, 3, 4]，它算出 1+3=4，正确答案是 2+4=6。

循环第一步是先生成测试。测试状态机产出一个独立脚本，用 assert 断言正确行为：

- 第一轮提出 `assert sum_evens([]) == 0`。空列表下任何实现都返回 0，这个测试在未修复代码与参考实现上都通过——它复现不了 bug，被拒绝，再来一轮。
- 第二轮提出 `assert sum_evens([1, 2, 3, 4]) == 6`。在未修复代码上算出 4，断言失败（退出码非 0），复现了 issue；在参考实现上 6 等于 6，通过。测试被接受。

第二步是编辑与测试互纠。编辑状态机读取当前源码，修改后跑这个测试：

- 第一轮编辑把条件改成 `x % 2 == 2`。对任何整数 x，取余结果只有 0 或 1，这个条件恒为假，total 恒为 0。跑测试：0 不等于 6，退出码非 0，输出尾部带着断言失败的回溯。
- 把"exit 非 0 + 输出尾部"当作反馈喂回。第二轮编辑把条件改成 `x % 2 == 0`。跑测试：得到 6，退出码 0，输出 PASS。循环终止，源码被接受。

整个过程就是读代码、改、跑测试、看输出、再改。每一步 feedback 都是真实的执行结果，模型据此修正下一步。测试与编辑拆成两个状态机，各有独立接受判据：测试要求两侧检查（未修复上失败、参考实现上通过），编辑要求测试通过。拆开的好处是测试先被验证可靠，编辑只对已知可靠的判官作答，两个状态机的错误不会相互污染。

In [ ]:
def extract_python(text):
    """从回复里提取 ```python ... ``` 代码块，没有则返回空串。"""
    start = text.find("```python")
    if start == -1:
        return ""
    start = text.find("\n", start) + 1
    end = text.find("```", start)
    return text[start:end] if end != -1 else text[start:]

def propose_test(issue, client, round_no):
    """请求 LLM 生成一段能复现 issue 的测试脚本。

    真实 API 演示下返回脚本化候选：第一轮给一个两版代码都通过的
    弱测试（抓不到 bug），第二轮给能复现 bug 的强测试。
    """
    if False:
        scripted = [
            "from buggy import sum_evens\n"
            "assert sum_evens([]) == 0\nprint('PASS')",
            "from buggy import sum_evens\n"
            "assert sum_evens([1, 2, 3, 4]) == 6\nprint('PASS')",
        ]
        return scripted[min(round_no, len(scripted) - 1)]
    prompt = ("仓库 buggy.py 的函数 sum_evens 有缺陷。请写一段独立的 "
              "Python 测试脚本，在未修复的代码上必须断言失败。\n" + issue)
    reply = client.chat([{"role": "user", "content": prompt}])
    return extract_python(reply) or ""

def generate_test(issue, repo_dir, client, max_rounds=3):
    """测试状态机：迭代出能复现 issue 的测试脚本。

    接受标准是两侧检查：脚本在未修复代码上必须失败（复现 bug），
    在参考实现上必须通过。返回 (是否接受, 测试脚本, 每轮轨迹)。
    """
    trace = []
    test = ""
    for r in range(max_rounds):
        test = propose_test(issue, client, r)
        code_buggy, _ = run_test(repo_dir, test)
        with open(os.path.join(repo_dir, "buggy.py"), "w") as f:
            f.write(FIXED)
        code_fixed, _ = run_test(repo_dir, test)
        with open(os.path.join(repo_dir, "buggy.py"), "w") as f:
            f.write(BUGGY)
        reproduced = code_buggy != 0
        verified = code_fixed == 0
        trace.append({"round": r, "reproduced": reproduced,
                      "verified": verified})
        if reproduced and verified:
            return True, test, trace
    return False, test, trace

print("测试状态机已就绪：extract_python / propose_test / generate_test")

In [ ]:
def propose_edit(current_source, feedback, client, round_no):
    """请求 LLM 针对当前源码与执行反馈生成修改后的源码。

    真实 API 演示下返回脚本化轨迹：第一轮给一个仍带 bug 的修改
    （把判定条件改成恒假），第二轮给正确修复。
    """
    if False:
        scripted = [
            current_source.replace("x % 2 == 1", "x % 2 == 2"),
            FIXED,
        ]
        return scripted[min(round_no, len(scripted) - 1)]
    prompt = ("当前源码：\n" + current_source + "\n执行反馈：\n" + feedback
              + "\n请输出修复后完整的 buggy.py 源码，放在 ```python 块里。")
    reply = client.chat([{"role": "user", "content": prompt}])
    new_source = extract_python(reply)
    return new_source if new_source else current_source

def repair_loop(issue, repo_dir, test, client, max_rounds=4):
    """编辑状态机：读取源码、修改、跑测试、拿反馈、重试。

    测试脚本已由测试状态机产出并接受。每轮把 exit code 与输出
    当作反馈喂回模型。返回 (最终源码, 是否修好, 每轮轨迹)。
    """
    with open(os.path.join(repo_dir, "buggy.py")) as f:
        source = f.read()
    trace = []
    feedback = "初始状态，还没有执行过测试。"
    for r in range(max_rounds):
        new_source = propose_edit(source, feedback, client, r)
        with open(os.path.join(repo_dir, "buggy.py"), "w") as f:
            f.write(new_source)
        code, tail = run_test(repo_dir, test)
        trace.append({"round": r, "exit": code, "tail": tail})
        if code == 0:
            return new_source, True, trace
        feedback = "exit=%d，输出：%s" % (code, " | ".join(tail))
        source = new_source          # 把上一轮的修改作为下一轮的起点
    return new_source, False, trace

print("编辑状态机已就绪：propose_edit / repair_loop")

In [ ]:
issue = ("sum_evens(nums) 应该返回列表中所有偶数的和，"
         "但当前实现把奇数也算进去了。")
ok_test, test, test_trace = generate_test(issue, repo_dir, client)
print("测试状态机：接受 = %s" % ok_test)
print("测试脚本：\n%s" % test)
for step in test_trace:
    print("  第 %d 轮  复现 bug=%s  参考实现通过=%s"
          % (step["round"], step["reproduced"], step["verified"]))

fixed, solved, edit_trace = repair_loop(issue, repo_dir, test, client)
print("编辑状态机：修好 = %s" % solved)
for step in edit_trace:
    print("  第 %d 轮  exit=%s  输出=%s"
          % (step["round"], step["exit"], step["tail"]))

In [ ]:
# 用测试状态机没见过的输入做独立验证
oracle_test = ("from buggy import sum_evens\n"
               "assert sum_evens([]) == 0\n"
               "assert sum_evens([0, 2, 4, 6]) == 12\n"
               "assert sum_evens([1, 3, 5]) == 0\n"
               "assert sum_evens([1, 2, 3, 4, 5]) == 6\n"
               "print('ORACLE PASS')\n")
code, tail = run_test(repo_dir, oracle_test)
print("独立 oracle 验证 exit=%d，输出=%s" % (code, tail))
print("最终源码：\n" + fixed)
print("关键观察：两个状态机各管一半——测试负责'复现并判定'，"
      "编辑负责'修改并提交'；真实 API 演示下产物来自脚本化轨迹，"
      "真实 API 下由模型生成，循环骨架不变。")

## 小结

这一讲把 Agent 从合成任务搬到真实软件工程任务，沿"找上下文、生成候选、选择答案"的脚手架走了一遍：

- [ ] 修一个 issue 可以切成三段，各段有独立度量：上下文召回、生成覆盖率、选择得分
- [ ] 覆盖率随采样数近似对数线性增长，前几次采样收益最大
- [ ] 相同总预算下，串行修复与并行采样的不同分配得到的覆盖率相近；预算紧张时修复更划算
- [ ] 覆盖率是生成的天花板，选择方法决定从天花板上收回多少
- [ ] fast_p 用一个阈值同时编码正确性与加速比，p 是评测的旋钮，调高就变难
- [ ] 执行反馈与 profiler 反馈抬高每轮成功率，迭代优化强于同预算的重复采样
- [ ] Agent 的意图必须与接口的实现语义对齐，否则指标漂移（construct drift）
- [ ] 迷你 SWE 循环 = 测试状态机（两侧检查）+ 编辑状态机（修改→测试→反馈→重试）

## 作业

> 可以让 AI 帮忙解释思路，但不建议直接让 AI "做完这道题"。

**作业 1：轨迹修复的成功率**

补全 trajectory_success，计算单条轨迹做 r 轮串行修复后的解出概率。

```python
def trajectory_success(p, r, eta):
    """单条轨迹：初次生成正确概率 p，失败后每轮有概率 eta 补对。"""
    return ____                          # 补全

assert abs(trajectory_success(0.2, 2, 0.5) - (1 - 0.8 * 0.5 ** 2)) < 1e-9
assert abs(trajectory_success(0.2, 0, 0.5) - 0.2) < 1e-12
print("轨迹修复模型正确：初次失败后每轮仍有 eta 的机会补对。")
```

小提示：初次生成失败的概率是 (1-p)；之后每轮都以 (1-eta) 的概率继续失败，r 轮全失败的概率是 $(1-p)(1-eta)^r$，成功概率用 1 减。

**作业 2：fast_p 的阈值旋钮**

补全 fast_p，返回正确且加速比大于 p 的候选占比。

```python
def fast_p(cands, p):
    """cands：形如 (is_correct, speedup) 的列表。"""
    correct = np.array([c[0] for c in cands], dtype=bool)
    speedup = np.array([c[1] for c in cands])
    return ____                          # 补全

cands = [(True, 3.0), (True, 0.5), (False, 4.0), (True, 1.2)]
assert abs(fast_p(cands, 0) - 0.75) < 1e-9
assert abs(fast_p(cands, 1) - 0.5) < 1e-9
assert fast_p(cands, 2) <= fast_p(cands, 1)
print("fast_p 正确：阈值越高，'对且快'的候选越少。")
```

小提示：用逐元素的 & 连接两个布尔条件，再取 mean——注意是 & 而不是 and。

**作业 3：用测试投票选择候选**

补全 vote_select：passes 是 (T, C) 的布尔矩阵，T 个生成测试对 C 条候选的通过情况；lengths 是每条候选补丁的长度。选出通过测试最多、平手时补丁最短的候选。

```python
def vote_select(passes, lengths):
    """返回通过测试最多、平手时补丁最短的候选下标。"""
    votes = passes.sum(axis=0)
    max_votes = ____                          # 补全：最高票数
    top = np.where(votes == max_votes)[0]
    return top[np.argmin(____)]               # 补全：高票组里取补丁最短者

passes = np.array([[1, 1, 0],
                   [1, 1, 0],
                   [0, 0, 1]])
lengths = np.array([40, 25, 60])
assert vote_select(passes, lengths) == 1
print("投票选择器正确：第 1 条候选两票居首，且补丁更短。")
```

小提示：`passes.sum(axis=0)` 是每列的通过数；先在通过数最大的子集里，再按补丁长度取最小。

## 参考资料

- Ehrlich et al., [CodeMonkeys: Scaling Test-Time Compute for Software Engineering](https://arxiv.org/abs/2501.14723), 2025 — 本讲第一主力：三段式分解加两个背靠背状态机，SWE-bench Verified 57.4%
- Ouyang et al., [KernelBench: Can LLMs Write Efficient GPU Kernels?](https://arxiv.org/abs/2502.10517), 2025 — fast_p 指标与生成-编译-执行-profiler 反馈环，R1 的 Level 2 fast_1 从 36% 升到 72%
- Trivedi & Nord, [On the Need to Align Intent and Implementation in Uncertainty Quantification for Machine Learning](https://arxiv.org/abs/2506.03037), 2025 — construct drift 与"先声明推理链再做宣称"的诊断纪律
- Wei et al., [Improving Parallel Program Performance with LLM Optimizers via Agent-System Interfaces](https://arxiv.org/abs/2410.15625), ICML 2025 — Agent-System 接口的落地方案：DSL 加 AutoGuide，10 次迭代超过 OpenTuner 千次迭代
- Jimenez et al., [SWE-bench: Can Language Models Resolve Real-World GitHub Issues?](https://arxiv.org/abs/2310.06770), 2023 — 真实 GitHub issue 评测基准，SWE-bench Verified 的出处
- Brown et al., [Large Language Monkeys: Scaling Inference Compute with Repeated Sampling](https://arxiv.org/abs/2407.21787), 2024 — "覆盖率随采样近似对数线性增长"的出处，CodeMonkeys 的直接前作
- Yang et al., [SWE-Agent: Agent-Computer Interfaces Enable Automated Software Engineering](https://arxiv.org/abs/2405.15793), 2024 — 强调 agent-computer 接口的另一个 SWE 框架，与第 3 节互补
- Xia et al., [Agentless: Demystifying LLM-based Software Engineering Agents](https://arxiv.org/abs/2407.01489), 2024 — 定位-修复-验证三段式，CodeMonkeys 的对照系
- Tillet et al., [Triton: An Intermediate Language and Compiler for Tiled Neural Network Computations](https://openreview.net/forum?id=RR1J7pJwqB), 2019 — KernelBench 允许使用的高层内核语言，隐式处理 tiling 与共享内存
- Dao et al., [FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness](https://arxiv.org/abs/2205.14135), 2022 — 人类写高效内核的标杆，KernelBench 反复引用